# DiabetesCheck: diabetes screening without a blood test

**Veritas AI International High School Hackathon 2026 · Track: AI + Health (our own idea)**
**Team:** Hima Shreyas Nagumalla and Nishant Garikota

**One-sentence pitch:** DiabetesCheck helps adults decide whether to get a blood-sugar test, the same job the CDC's paper risk quiz does, using only 8 quick survey questions, and checks whether it does better than that quiz.

**Live app:** https://jarvis21stcentury.github.io/DiabetesCheck/ · **Code:** https://github.com/Jarvis21stCentury/DiabetesCheck

**How to run:** Runtime → Run all. Takes about 8 minutes. No API keys or uploads needed.

*This project is a prototype and is not medical advice.*

**What's in this notebook**
1. Load and look at the data · 2. Split · 3. Baselines, including the **CDC Prediabetes Risk Test** (our added comparison) · 4. Models · 5. A fair comparison at the same catch rate · 6. Choosing the cut-off · 7. Where it fails and who it misses · 8. Final test · 9. Demo · 10. Results and limitations

Cells marked **← OUR DECISION** are choices we made, with our reasons written below them.

## Toolkit: helper functions we use throughout

In [1]:
import numpy as np, pandas as pd
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, mean_absolute_error, confusion_matrix)

results = []  # one row per model; this becomes our results table

def evaluate(name, y_true, y_pred, y_score=None, task="classification", split="validation"):
    """Score one model and add a row to the results table."""
    row = {"model": name, "split": split, "n": len(y_true)}
    if task == "classification":   # assumes labels are 0/1
        row["precision"] = precision_score(y_true, y_pred, zero_division=0)  # of our flags, share that were right
        row["recall"] = recall_score(y_true, y_pred, zero_division=0)        # of real positives, share we caught
        row["F1"] = f1_score(y_true, y_pred, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        row["false alarms"], row["missed"] = fp, fn   # reported separately, as the health track asks
        if y_score is not None:   # probabilities/scores, not 0/1 labels
            row["AUROC"] = roc_auc_score(y_true, y_score)
            row["PR-AUC"] = average_precision_score(y_true, y_score)
    else:
        row["MAE"] = mean_absolute_error(y_true, y_pred)
    results.append(row)
    return pd.DataFrame(results).round(3)

In [2]:
def show_failures(X, y_true, y_pred, n=5, task="classification"):
    """Classification: n examples it got wrong. Regression: the n biggest misses."""
    df = pd.DataFrame(X).copy()
    df["truth"], df["predicted"] = np.asarray(y_true), np.asarray(y_pred)
    if task == "classification":
        return df[df["truth"] != df["predicted"]].head(n)
    df["error"] = (df["truth"] - df["predicted"]).abs()
    return df.sort_values("error", ascending=False).head(n)

def by_group(groups, y_true, y_pred, task="classification"):
    """Score the model separately for each group (e.g. age band), with group sizes."""
    df = pd.DataFrame({"group": np.asarray(groups), "truth": np.asarray(y_true), "pred": np.asarray(y_pred)})
    rows = []
    for g, d in df.groupby("group"):
        if task == "classification":
            rows.append({"group": g, "n": len(d), "real cases": int(d.truth.sum()),
                         "recall": recall_score(d.truth, d.pred, zero_division=0),
                         "precision": precision_score(d.truth, d.pred, zero_division=0),
                         "positive rate": d.pred.mean()})   # how often the model says "yes" for this group
        else:
            rows.append({"group": g, "n": len(d), "MAE": mean_absolute_error(d.truth, d.pred)})
    return pd.DataFrame(rows).round(3)

In [3]:
def threshold_table(y_true, y_score, thresholds=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7)):
    """For each cut-off: how many we flag, how many real cases we catch, and how many false alarms."""
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    rows = []
    for t in thresholds:
        pred = (y_score >= t).astype(int)     # flag as "yes" if the score clears the cut-off
        rows.append({"cut-off": t, "flagged": int(pred.sum()),
                     "share flagged": pred.mean(),
                     "recall": recall_score(y_true, pred, zero_division=0),
                     "precision": precision_score(y_true, pred, zero_division=0),
                     "false alarms": int(((pred == 1) & (y_true == 0)).sum()),
                     "missed": int(((pred == 0) & (y_true == 1)).sum())})
    return pd.DataFrame(rows).round(3)

def cutoff_for_recall(y_true, y_score, target):
    """The highest cut-off that still catches at least `target` share of the real cases."""
    pos = np.sort(np.asarray(y_score)[np.asarray(y_true) == 1])[::-1]   # real cases' scores, highest first
    k = int(np.ceil(target * len(pos)))                                  # how many real cases we must catch
    return pos[k - 1]

## 1. Load the data

**CDC Diabetes Health Indicators** (UCI Machine Learning Repository, dataset 891): 253,680 adults who answered the CDC's 2015 BRFSS health survey. Each row has 21 survey answers. The label `Diabetes_binary` is 1 if the person has been told they have diabetes or prediabetes, and 0 if not.

In [4]:
!pip install -q ucimlrepo
from ucimlrepo import fetch_ucirepo

cdc = fetch_ucirepo(id=891)
X = cdc.data.features                        # the 21 survey answers (what goes in)
y = cdc.data.targets["Diabetes_binary"]      # 1 = diabetes or prediabetes, 0 = neither (what we predict)

print(X.shape)
print(list(X.columns))

(253680, 21)
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


## 2. Look at the data first

In [5]:
print("Share with diabetes/prediabetes:", round(y.mean(), 3))
print("Exact duplicate rows:", X.assign(Diabetes_binary=y).duplicated().sum())
print(X[["BMI", "Age", "Income", "Sex", "HighBP", "GenHlth"]].describe().round(1))
cdc.variables[["name", "description"]]

Share with diabetes/prediabetes: 0.139
Exact duplicate rows: 24206
            BMI       Age    Income       Sex    HighBP   GenHlth
count  253680.0  253680.0  253680.0  253680.0  253680.0  253680.0
mean       28.4       8.0       6.1       0.4       0.4       2.5
std         6.6       3.1       2.1       0.5       0.5       1.1
min        12.0       1.0       1.0       0.0       0.0       1.0
25%        24.0       6.0       5.0       0.0       0.0       2.0
50%        27.0       8.0       7.0       0.0       0.0       2.0
75%        31.0      10.0       8.0       1.0       1.0       3.0
max        98.0      13.0       8.0       1.0       1.0       5.0


,name,description
0,ID,Patient ID
1,Diabetes_binary,0 = no diabetes 1 = prediabetes or diabetes
2,HighBP,0 = no high BP 1 = high BP
3,HighChol,0 = no high cholesterol 1 = high cholesterol
4,CholCheck,0 = no cholesterol check in 5 years 1 = yes ch...
5,BMI,Body Mass Index
6,Smoker,Have you smoked at least 100 cigarettes in you...
7,Stroke,(Ever told) you had a stroke. 0 = no 1 = yes
8,HeartDiseaseorAttack,coronary heart disease (CHD) or myocardial inf...
9,PhysActivity,physical activity in past 30 days - not includ...


**What we learned**
- About **14%** of people are "yes", so the classes are lopsided. Saying "no" to everyone would already be 86% accurate while catching nobody, so we report **recall** (share of real cases caught), **precision** (share of flags that were right), **missed cases** and **false alarms** instead of accuracy.
- Age, Income and General health are **codes**, not real values: Age 1 = 18–24 … 13 = 80+; Income 1 = under \$10k … 5 = under \$35k … 8 = \$75k+; General health 1 = excellent … 5 = poor; Sex 0 = female, 1 = male.
- About 24,000 rows are exact duplicates. **← OUR DECISION:** we keep them. Most answers are yes/no or an age group, so two real people can easily give exactly the same answers. Deleting duplicates would delete real people and change who is in the data.

## 3. Split into train, validation and test

- **Train (60%)**: the model learns from these people.
- **Validation (20%)**: we compare models and choose settings on these.
- **Test (20%)**: locked away. We score it **once**, at the end, so it's an honest final grade.

`stratify=y` keeps the same share of "yes" in all three. `random_state=42` makes the split the same every run.

In [6]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)            # 20% locked away as the test set
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)  # 25% of 80% = 20%

for name, part in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name}: {len(part):,} people, {part.mean():.1%} yes")

train: 152,208 people, 13.9% yes
validation: 50,736 people, 13.9% yes
test: 50,736 people, 13.9% yes


## 4. Baselines: the guesses our model has to beat

1. **Always "no"**: the laziest possible guess.
2. **A simple rule**: send someone for a test if their BMI is 30+ **or** they have high blood pressure.

In [7]:
always_no = np.zeros(len(y_val), dtype=int)
evaluate("Baseline: always 'no'", y_val, always_no)

rule_val = ((X_val["BMI"] >= 30) | (X_val["HighBP"] == 1)).astype(int)
evaluate("Baseline: rule (BMI 30+ or high BP)", y_val, rule_val)

,model,split,n,precision,recall,F1,false alarms,missed
0,Baseline: always 'no',validation,50736,0.00,0.000,0.000,0,7069
1,Baseline: rule (BMI 30+ or high BP),validation,50736,0.21,0.868,0.338,23088,930


### 4b. Our added comparison: the CDC Prediabetes Risk Test

Doctors and pharmacies already use a free paper quiz from the **CDC and the American Diabetes Association** to decide who should get a blood-sugar test. **Our question: does our model actually beat the quiz people already use?**

Official scoring (CDC Prediabetes Risk Test; a score of **5 or more** = high risk):

| Question | Points | Our column |
|---|---|---|
| Age: under 40 / 40–49 / 50–59 / 60+ | 0 / 1 / 2 / 3 | `Age` codes 1–4 / 5–6 / 7–8 / 9–13 |
| Man | 1 | `Sex` = 1 |
| High blood pressure | 1 | `HighBP` = 1 |
| Not physically active | 1 | `PhysActivity` = 0 |
| Weight for height | 1 / 2 / 3 | `BMI` 25–29 / 30–39 / 40+ |
| Family history of diabetes | 1 | **not in our data** |
| Gestational diabetes (women) | 1 | **not in our data** |

**How we turned the weight chart into BMI:** we checked the chart's own numbers. For example, at 5'6" the 1-point column starts at 155 lb (BMI 25.0), the 2-point column at 186 lb (BMI 30.0), and the 3-point column at 247 lb (BMI 39.9). So the bands match BMI 25, 30 and 40.

**Limitation:** our version is missing 2 questions, so real scores would be up to 2 points higher. We report the official cut-off (5+) and also 4+ to account for that.

Source: CDC Prediabetes Risk Test, https://www.cdc.gov/diabetes/prevention/pdf/Prediabetes-Risk-Test-Final.pdf (adapted from Bang et al., *Annals of Internal Medicine*, 2009).

In [8]:
def cdc_quiz_points(df):
    """Partial CDC Prediabetes Risk Test score for each person (family history and gestational diabetes not available)."""
    age = df["Age"]
    pts  = np.select([age <= 4, age <= 6, age <= 8], [0, 1, 2], default=3)   # under 40 / 40-49 / 50-59 / 60+
    pts += (df["Sex"] == 1).astype(int)                                      # man
    pts += (df["HighBP"] == 1).astype(int)                                   # high blood pressure
    pts += (df["PhysActivity"] == 0).astype(int)                             # not physically active
    pts += np.select([df["BMI"] >= 40, df["BMI"] >= 30, df["BMI"] >= 25], [3, 2, 1], default=0)  # weight
    return pd.Series(pts, index=df.index, name="CDC quiz points")

quiz_val = cdc_quiz_points(X_val)
print(quiz_val.value_counts().sort_index())   # how many people got each score

evaluate("CDC risk quiz (partial), score 5+", y_val, (quiz_val >= 5).astype(int), quiz_val)
evaluate("CDC risk quiz (partial), score 4+", y_val, (quiz_val >= 4).astype(int), quiz_val)

CDC quiz points
0    1372
1    3072
2    5077
3    7562
4    9629
5    9940
6    8098
7    4592
8    1256
9     138
Name: count, dtype: int64


,model,split,n,precision,recall,F1,false alarms,missed,AUROC,PR-AUC
0,Baseline: always 'no',validation,50736,0.000,0.000,0.000,0,7069,NaN,NaN
1,Baseline: rule (BMI 30+ or high BP),validation,50736,0.210,0.868,0.338,23088,930,NaN,NaN
2,"CDC risk quiz (partial), score 5+",validation,50736,0.237,0.807,0.367,18320,1365,0.758,0.298
3,"CDC risk quiz (partial), score 4+",validation,50736,0.195,0.929,0.322,27088,504,0.758,0.298


## 5. Model 1: logistic regression on all 21 questions

Logistic regression learns how much each answer pushes the risk up or down, and outputs a probability between 0 and 1.
- `StandardScaler` puts all columns on the same scale, so BMI's bigger numbers don't drown out yes/no answers. It sits **inside** the pipeline, so it only learns from training data.
- `class_weight="balanced"` counts a missed "yes" as more costly, because only 14% of people are "yes".

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model1 = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=1000, class_weight="balanced"))
model1.fit(X_train, y_train)                          # learn from TRAIN only
score1 = model1.predict_proba(X_val)[:, 1]            # confidence 0-1 for each person
evaluate("Model 1: logistic regression (21 questions)", y_val, (score1 >= 0.5).astype(int), score1)

,model,split,n,precision,recall,F1,false alarms,missed,AUROC,PR-AUC
0,Baseline: always 'no',validation,50736,0.000,0.000,0.000,0,7069,NaN,NaN
1,Baseline: rule (BMI 30+ or high BP),validation,50736,0.210,0.868,0.338,23088,930,NaN,NaN
2,"CDC risk quiz (partial), score 5+",validation,50736,0.237,0.807,0.367,18320,1365,0.758,0.298
3,"CDC risk quiz (partial), score 4+",validation,50736,0.195,0.929,0.322,27088,504,0.758,0.298
4,Model 1: logistic regression (21 questions),validation,50736,0.311,0.762,0.442,11935,1685,0.821,0.400


## 6. Short survey: can 8 quick questions do the job?

Nobody wants to answer 21 questions. These 8 can be answered in under a minute, and they're what our app asks.

In [10]:
SHORT = ["BMI", "Age", "Sex", "HighBP", "HighChol", "PhysActivity", "GenHlth", "Smoker"]

model_short = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000, class_weight="balanced"))
model_short.fit(X_train[SHORT], y_train)              # same training people, only 8 columns
score_short = model_short.predict_proba(X_val[SHORT])[:, 1]
evaluate("Model 1b: logistic regression (8 questions)", y_val, (score_short >= 0.5).astype(int), score_short)

,model,split,n,precision,recall,F1,false alarms,missed,AUROC,PR-AUC
0,Baseline: always 'no',validation,50736,0.000,0.000,0.000,0,7069,NaN,NaN
1,Baseline: rule (BMI 30+ or high BP),validation,50736,0.210,0.868,0.338,23088,930,NaN,NaN
2,"CDC risk quiz (partial), score 5+",validation,50736,0.237,0.807,0.367,18320,1365,0.758,0.298
3,"CDC risk quiz (partial), score 4+",validation,50736,0.195,0.929,0.322,27088,504,0.758,0.298
4,Model 1: logistic regression (21 questions),validation,50736,0.311,0.762,0.442,11935,1685,0.821,0.400
5,Model 1b: logistic regression (8 questions),validation,50736,0.306,0.759,0.436,12189,1704,0.816,0.392


## 7. Model 2: gradient boosting

Gradient boosting builds many small decision trees, each one fixing the mistakes of the ones before. It can pick up combinations (like "older **and** inactive") that logistic regression can't. Trees don't need scaling.

In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier

model2 = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
model2.fit(X_train, y_train)                          # TRAIN only
score2 = model2.predict_proba(X_val)[:, 1]
evaluate("Model 2: gradient boosting (21 questions)", y_val, (score2 >= 0.5).astype(int), score2)

model2_short = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
model2_short.fit(X_train[SHORT], y_train)
score2_short = model2_short.predict_proba(X_val[SHORT])[:, 1]
evaluate("Model 2b: gradient boosting (8 questions)", y_val, (score2_short >= 0.5).astype(int), score2_short)

,model,split,n,precision,recall,F1,false alarms,missed,AUROC,PR-AUC
0,Baseline: always 'no',validation,50736,0.000,0.000,0.000,0,7069,NaN,NaN
1,Baseline: rule (BMI 30+ or high BP),validation,50736,0.210,0.868,0.338,23088,930,NaN,NaN
2,"CDC risk quiz (partial), score 5+",validation,50736,0.237,0.807,0.367,18320,1365,0.758,0.298
3,"CDC risk quiz (partial), score 4+",validation,50736,0.195,0.929,0.322,27088,504,0.758,0.298
4,Model 1: logistic regression (21 questions),validation,50736,0.311,0.762,0.442,11935,1685,0.821,0.400
5,Model 1b: logistic regression (8 questions),validation,50736,0.306,0.759,0.436,12189,1704,0.816,0.392
6,Model 2: gradient boosting (21 questions),validation,50736,0.303,0.790,0.438,12821,1484,0.828,0.432
7,Model 2b: gradient boosting (8 questions),validation,50736,0.300,0.783,0.433,12948,1533,0.821,0.421


### 7b. Are these the best 8 questions?

We picked our 8 questions by hand. Here we test every single swap: take out one of our 8 and put in one of the other 13. We use logistic regression because it's fast, and keep a swap only if it clearly helps (PR-AUC up by more than 0.005).

In [12]:
def quick_prauc(cols):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
    m.fit(X_train[cols], y_train)
    return average_precision_score(y_val, m.predict_proba(X_val[cols])[:, 1])

base = quick_prauc(SHORT)
swaps = []
for out_q in SHORT:
    for in_q in [c for c in X.columns if c not in SHORT]:
        cols = [c for c in SHORT if c != out_q] + [in_q]
        swaps.append({"remove": out_q, "add": in_q, "PR-AUC": quick_prauc(cols)})
swaps = pd.DataFrame(swaps).sort_values("PR-AUC", ascending=False)
swaps["change"] = swaps["PR-AUC"] - base
print(f"Our 8 questions: PR-AUC {base:.3f}")
swaps.head(8).round(3)

Our 8 questions: PR-AUC 0.392


,remove,add,PR-AUC,change
96,Smoker,HvyAlcoholConsump,0.395,0.003
70,PhysActivity,HvyAlcoholConsump,0.395,0.003
93,Smoker,HeartDiseaseorAttack,0.394,0.002
67,PhysActivity,HeartDiseaseorAttack,0.394,0.002
91,Smoker,CholCheck,0.394,0.002
65,PhysActivity,CholCheck,0.393,0.001
103,Smoker,Income,0.393,0.001
101,Smoker,DiffWalk,0.393,0.001


In [13]:
add_one = []
for extra in [c for c in X.columns if c not in SHORT]:
    m = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
    m.fit(X_train[SHORT + [extra]], y_train)
    add_one.append({"9th question": extra,
                    "PR-AUC": average_precision_score(y_val, m.predict_proba(X_val[SHORT + [extra]])[:, 1])})
print("Our 8 questions: PR-AUC", round(average_precision_score(y_val, score2_short), 3))
pd.DataFrame(add_one).sort_values("PR-AUC", ascending=False).round(3).head(5)

Our 8 questions: PR-AUC 0.421


,9th question,PR-AUC
11,Education,0.424
5,HvyAlcoholConsump,0.424
12,Income,0.424
2,HeartDiseaseorAttack,0.424
0,CholCheck,0.423


**← OUR DECISION:** if a swap gains more than about 0.005, we can change the `SHORT` list in section 6 and Run all again. Smaller gains are probably noise, so we keep our original 8, which are also quick to answer. 

**What we decided:** no swap or 9th question helped by more than 0.003 PR-AUC (the best, adding education or heavy drinking, moved it from 0.421 to 0.424). That's below our 0.005 noise bar, so we kept the original 8 questions.

### 7c. Which questions matter most?

**Permutation importance:** shuffle one question's answers across people and see how much worse the model gets. If shuffling a question hurts a lot, the model relies on it.

In [14]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(model2_short, X_val[SHORT], y_val, scoring="average_precision",
                             n_repeats=3, random_state=42)
importance = pd.DataFrame({"question": SHORT, "PR-AUC drop when shuffled": imp.importances_mean})
importance.sort_values("PR-AUC drop when shuffled", ascending=False).round(4)

,question,PR-AUC drop when shuffled
6,GenHlth,0.1070
0,BMI,0.0959
1,Age,0.0474
3,HighBP,0.0357
4,HighChol,0.0297
2,Sex,0.0039
7,Smoker,0.0008
5,PhysActivity,0.0004


**What drives the score:** General health, BMI and age matter most (shuffling them drops PR-AUC by 0.107, 0.096 and 0.047), followed by high blood pressure and high cholesterol. That makes sense: weight, age and blood pressure are also on the CDC's own risk quiz. Smoking and exercise add almost nothing once the other answers are known.

## 8. A fair comparison: same catch rate, fewer wasted tests?

Comparing models at the default 0.5 cut-off isn't fair, because each one catches a different share of real cases. So we **fix the catch rate** to match the CDC quiz and the simple rule, then count how many people each approach sends for a test. **Fewer flags at the same catch rate = fewer unnecessary blood tests.**

In [15]:
def same_catch_rate_table(target_name, target_pred):
    """Every model set to catch the same share of real cases as `target_pred`."""
    target_recall = recall_score(y_val, target_pred)
    rows = [{"approach": target_name, "recall": target_recall, "flagged": int(target_pred.sum()),
             "false alarms": int(((target_pred == 1) & (y_val == 0)).sum()),
             "precision": precision_score(y_val, target_pred)}]
    for name, s in [("Model 1 (21 q)", score1), ("Model 1b (8 q)", score_short),
                    ("Model 2 (21 q)", score2), ("Model 2b (8 q)", score2_short)]:
        cut = cutoff_for_recall(y_val, s, target_recall)
        p = (s >= cut).astype(int)
        rows.append({"approach": f"{name} at cut-off {cut:.2f}", "recall": recall_score(y_val, p),
                     "flagged": int(p.sum()), "false alarms": int(((p == 1) & (y_val == 0)).sum()),
                     "precision": precision_score(y_val, p)})
    return pd.DataFrame(rows).round(3)

quiz_pred_val = (quiz_val >= 5).astype(int)
print("Matched to the CDC quiz (score 5+):")
display(same_catch_rate_table("CDC risk quiz, score 5+", quiz_pred_val))
print("Matched to the simple rule (BMI 30+ or high BP):")
display(same_catch_rate_table("Rule: BMI 30+ or high BP", rule_val))
display(same_catch_rate_table("CDC risk quiz, score 4+", (quiz_val >= 4).astype(int)))

Matched to the CDC quiz (score 5+):


,approach,recall,flagged,false alarms,precision
0,"CDC risk quiz, score 5+",0.807,24024,18320,0.237
1,Model 1 (21 q) at cut-off 0.45,0.807,19373,13669,0.294
2,Model 1b (8 q) at cut-off 0.45,0.807,19790,14086,0.288
3,Model 2 (21 q) at cut-off 0.48,0.807,19259,13555,0.296
4,Model 2b (8 q) at cut-off 0.48,0.807,19669,13962,0.290


Matched to the simple rule (BMI 30+ or high BP):


,approach,recall,flagged,false alarms,precision
0,Rule: BMI 30+ or high BP,0.868,29227,23088,0.210
1,Model 1 (21 q) at cut-off 0.38,0.868,22987,16848,0.267
2,Model 1b (8 q) at cut-off 0.38,0.868,23465,17326,0.262
3,Model 2 (21 q) at cut-off 0.40,0.868,22704,16565,0.270
4,Model 2b (8 q) at cut-off 0.40,0.868,23273,17134,0.264


,approach,recall,flagged,false alarms,precision
0,"CDC risk quiz, score 4+",0.929,33653,27088,0.195
1,Model 1 (21 q) at cut-off 0.28,0.929,28130,21565,0.233
2,Model 1b (8 q) at cut-off 0.28,0.929,28605,22040,0.230
3,Model 2 (21 q) at cut-off 0.28,0.929,28162,21597,0.233
4,Model 2b (8 q) at cut-off 0.28,0.929,28833,22268,0.228


### 8b. Is the win real? Five different splits

One split could be lucky. Here we redo the whole comparison with 5 different random splits and check whether our 8-question model beats the CDC quiz every time.

In [16]:
rows = []
for seed in [1, 2, 3, 4, 5]:
    Xa, Xb, ya, yb = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=seed)
    m = HistGradientBoostingClassifier(class_weight="balanced", random_state=42).fit(Xa[SHORT], ya)
    s = m.predict_proba(Xb[SHORT])[:, 1]
    q = cdc_quiz_points(Xb)
    q_pred = (q >= 5).astype(int)
    cut = cutoff_for_recall(yb, s, recall_score(yb, q_pred))          # match the quiz's catch rate
    rows.append({"split": seed, "model AUROC": roc_auc_score(yb, s), "quiz AUROC": roc_auc_score(yb, q),
                 "model false alarms": int(((s >= cut) & (yb == 0)).sum()),
                 "quiz false alarms": int(((q_pred == 1) & (yb == 0)).sum())})
robust = pd.DataFrame(rows)
robust["fewer wasted tests"] = 1 - robust["model false alarms"] / robust["quiz false alarms"]
display(robust.round(3))
print("Average:", robust[["model AUROC", "quiz AUROC", "fewer wasted tests"]].mean().round(3).to_dict())

,split,model AUROC,quiz AUROC,model false alarms,quiz false alarms,fewer wasted tests
0,1,0.823,0.762,14097,18316,0.230
1,2,0.824,0.765,14479,18231,0.206
2,3,0.825,0.764,14001,18193,0.230
3,4,0.822,0.757,14092,18381,0.233
4,5,0.827,0.761,13770,18377,0.251


Average: {'model AUROC': 0.824, 'quiz AUROC': 0.762, 'fewer wasted tests': 0.23}


(The test set is never used here: every split is made from the train + validation people only.)

**Our headline finding (validation):** At the same catch rate as the CDC quiz (80.7% of cases), our 8-question model sends **4,358 fewer people for unnecessary blood tests (13,962 vs 18,320), about 24% fewer.** Across 5 different random splits the saving was between 21% and 25% every time, so it isn't luck. The 8-question model was nearly as good as the 21-question one (13,962 vs 13,555 false alarms; AUROC 0.821 vs 0.828), so the short survey costs us very little.

## 9. Choosing the final model and cut-off

**← OUR DECISION.** Change these two lines based on the tables above, and write the reason in the Ledger.
- **Model:** we chose the 8-question model if it's close to the 21-question one, because people will actually answer 8 questions, and our app asks exactly those.
- **Cut-off:** lower means fewer missed cases but more tests. We picked the row that catches most cases without testing almost everyone.

In [17]:
final_name, final_model, final_cols = "Model 2b: gradient boosting (8 questions)", model2_short, SHORT   # ← OUR DECISION

final_score_val = final_model.predict_proba(X_val[final_cols])[:, 1]
threshold_table(y_val, final_score_val, thresholds=(0.3, 0.4, 0.5, 0.6, 0.7))

,cut-off,flagged,share flagged,recall,precision,false alarms,missed
0,0.3,27803,0.548,0.918,0.233,21311,577
1,0.4,23235,0.458,0.868,0.264,17101,935
2,0.5,18484,0.364,0.783,0.300,12948,1533
3,0.6,13829,0.273,0.678,0.347,9036,2276
4,0.7,9329,0.184,0.535,0.405,5548,3288


In [18]:
cutoff = 0.4                                              # ← OUR DECISION: replace with our chosen cut-off
final_pred_val = (final_score_val >= cutoff).astype(int)
evaluate(f"FINAL: {final_name} at cut-off {cutoff}", y_val, final_pred_val, final_score_val)

,model,split,n,precision,recall,F1,false alarms,missed,AUROC,PR-AUC
0,Baseline: always 'no',validation,50736,0.000,0.000,0.000,0,7069,NaN,NaN
1,Baseline: rule (BMI 30+ or high BP),validation,50736,0.210,0.868,0.338,23088,930,NaN,NaN
2,"CDC risk quiz (partial), score 5+",validation,50736,0.237,0.807,0.367,18320,1365,0.758,0.298
3,"CDC risk quiz (partial), score 4+",validation,50736,0.195,0.929,0.322,27088,504,0.758,0.298
4,Model 1: logistic regression (21 questions),validation,50736,0.311,0.762,0.442,11935,1685,0.821,0.400
5,Model 1b: logistic regression (8 questions),validation,50736,0.306,0.759,0.436,12189,1704,0.816,0.392
6,Model 2: gradient boosting (21 questions),validation,50736,0.303,0.790,0.438,12821,1484,0.828,0.432
7,Model 2b: gradient boosting (8 questions),validation,50736,0.300,0.783,0.433,12948,1533,0.821,0.421
8,FINAL: Model 2b: gradient boosting (8 question...,validation,50736,0.264,0.868,0.405,17101,935,0.821,0.421


**Why this cut-off:** At 0.40 the model catches **86.8% of real cases** (missing 935 of 7,069) while sending 45.8% of people for a test. That's the same catch rate as the simple BMI-or-blood-pressure rule, with about 6,000 fewer unneeded tests (17,101 vs 23,088). We cared more about missed cases than extra tests: a missed case goes untreated, while an extra blood test is quick and cheap. We chose this on validation data, before opening the test set.

### 9b. Is the screening score honest? (calibration)

Because of `class_weight="balanced"`, our scores run high: a score of 0.6 does **not** mean a 60% chance. So we never call it a probability. Instead we group validation people by score and count how many really had diabetes or prediabetes. The demo uses this table to say something true: *"among people with a similar score, about X in 10 had diabetes or prediabetes."*

In [19]:
score_bins = np.quantile(final_score_val, np.linspace(0, 1, 11))     # 10 groups with equal numbers of people
score_bins[0], score_bins[-1] = 0, 1
bin_id = np.clip(np.digitize(final_score_val, score_bins[1:-1]), 0, 9)
calib = pd.DataFrame({"group": bin_id, "score": final_score_val, "truth": np.asarray(y_val)}).groupby("group").agg(
    people=("truth", "size"), lowest_score=("score", "min"), average_score=("score", "mean"),
    really_positive=("truth", "mean"))
calib.round(3)

,people,lowest_score,average_score,really_positive
group,,,,
0,5074,0.003,0.019,0.005
1,5073,0.036,0.056,0.012
2,5063,0.080,0.113,0.024
3,5083,0.149,0.197,0.040
4,5063,0.245,0.298,0.064
5,5084,0.352,0.409,0.107
6,5075,0.464,0.517,0.150
7,5071,0.571,0.626,0.209
8,5076,0.682,0.730,0.305


**What this shows:** In every group the average score is far higher than the real rate. The top group averages 0.84, but only 48% of them really had diabetes or prediabetes. So the score ranks people well but is **not** a percent chance. That's why the app says "about X in 100 people with a similar score", using the `really_positive` column, and never shows the score as a percentage.

## 10. Where it fails

In [20]:
cm = confusion_matrix(y_val, final_pred_val, labels=[0, 1])
display(pd.DataFrame(cm, index=["really no", "really yes"], columns=["predicted no", "predicted yes"]))

cols = ["BMI", "Age", "Sex", "HighBP", "HighChol", "PhysActivity", "GenHlth", "Smoker", "truth", "predicted"]
print("Missed cases (really yes, predicted no):")
missed = show_failures(X_val, y_val, final_pred_val, n=len(y_val))
display(missed[missed["truth"] == 1].head(5)[cols])
print("False alarms (really no, predicted yes):")
display(missed[missed["truth"] == 0].head(5)[cols])

,predicted no,predicted yes
really no,26566,17101
really yes,935,6134


Missed cases (really yes, predicted no):


,BMI,Age,Sex,HighBP,HighChol,PhysActivity,GenHlth,Smoker,truth,predicted
140872,30,7,1,0,0,1,2,1,1,0
42201,31,8,0,1,0,1,2,1,1,0
85670,18,8,0,1,1,1,1,1,1,0
246421,30,9,0,0,0,1,2,0,1,0
207822,39,4,0,0,0,1,3,0,1,0


False alarms (really no, predicted yes):


,BMI,Age,Sex,HighBP,HighChol,PhysActivity,GenHlth,Smoker,truth,predicted
240275,38,9,1,1,1,1,3,1,0,1
190052,27,11,1,1,1,1,3,1,0,1
12256,29,10,1,1,0,0,4,1,0,1
232017,24,10,1,0,0,1,3,1,0,1
2339,31,9,1,1,1,1,3,1,0,1


**What the mistakes have in common:** The missed people mostly say they're active and in good or very good health, and most don't report high blood pressure, so the survey makes them look healthy even when their BMI is high. The false alarms are mostly older men with high blood pressure, high cholesterol and a smoking history: on paper they look exactly like high-risk people, but they didn't have diabetes. These are patterns in 10 rows, not proof.

## 11. Who does it miss? Results by age, sex and income

A screen that works well on average can still miss one group much more often. Health judges look for this specifically.

In [21]:
age_band = pd.cut(X_val["Age"], bins=[0, 5, 9, 13], labels=["18-44", "45-64", "65+"])   # codes 1-5, 6-9, 10-13
sex = X_val["Sex"].map({0: "female", 1: "male"})
income = np.where(X_val["Income"] <= 5, "under $35k", "$35k or more")                   # codes 1-5 vs 6-8

for title, groups in [("Age", age_band), ("Sex", sex), ("Income", income)]:
    print(f"\nOur model by {title}:")
    display(by_group(groups, y_val, final_pred_val))
    print(f"CDC quiz (5+) by {title}:")
    display(by_group(groups, y_val, quiz_pred_val))


Our model by Age:


,group,n,real cases,recall,precision,positive rate
0,18-44,11002,458,0.522,0.202,0.107
1,45-64,22062,2957,0.855,0.259,0.442
2,65+,17672,3654,0.922,0.274,0.696


CDC quiz (5+) by Age:


,group,n,real cases,recall,precision,positive rate
0,18-44,11002,458,0.221,0.149,0.062
1,45-64,22062,2957,0.776,0.220,0.473
2,65+,17672,3654,0.905,0.256,0.730



Our model by Sex:


,group,n,real cases,recall,precision,positive rate
0,female,28302,3660,0.857,0.263,0.422
1,male,22434,3409,0.879,0.266,0.503


... (2 outputs left out)


Our model by Income:


,group,n,real cases,recall,precision,positive rate
0,$35k or more,34009,3577,0.838,0.232,0.381
1,under $35k,16727,3492,0.898,0.305,0.615


CDC quiz (5+) by Income:


,group,n,real cases,recall,precision,positive rate
0,$35k or more,34009,3577,0.802,0.196,0.431
1,under $35k,16727,3492,0.812,0.302,0.561


**What we found:** Our model misses **young adults** most: it catches only 52% of cases in 18–44 year olds, versus 86% at 45–64 and 92% at 65+. The CDC quiz is even worse for that group (22%), so we still do better, but young adults are where a short survey works worst. We also do much better than the CDC quiz for women (86% vs 73% of cases caught).

## 12. Final test: run once, after the model and cut-off were frozen

Everything above used the validation set. This cell scores the frozen final model, the rule and the CDC quiz **once** on the test set, which nobody has looked at. Set `RUN_FINAL_TEST = True` only at the freeze.

In [22]:
RUN_FINAL_TEST = True      # ← flip to True ONCE at the Sunday 12 PM model freeze, then leave it True

if RUN_FINAL_TEST:
    rule_test = ((X_test["BMI"] >= 30) | (X_test["HighBP"] == 1)).astype(int)
    quiz_test = cdc_quiz_points(X_test)
    test_score = final_model.predict_proba(X_test[final_cols])[:, 1]
    evaluate("Rule (BMI 30+ or high BP)", y_test, rule_test, split="test")
    evaluate("CDC risk quiz (partial), score 5+", y_test, (quiz_test >= 5).astype(int), quiz_test, split="test")
    display(evaluate(f"FINAL: {final_name} at cut-off {cutoff}", y_test, (test_score >= cutoff).astype(int),
                     test_score, split="test").query("split == 'test'"))
else:
    print("Test set still locked. Run this at the Sunday 12 PM freeze.")

,model,split,n,precision,recall,F1,false alarms,missed,AUROC,PR-AUC
9,Rule (BMI 30+ or high BP),test,50736,0.208,0.867,0.336,23311,940,NaN,NaN
10,"CDC risk quiz (partial), score 5+",test,50736,0.237,0.812,0.367,18450,1331,0.757,0.295
11,FINAL: Model 2b: gradient boosting (8 question...,test,50736,0.263,0.867,0.404,17148,943,0.820,0.410


## 13. Demo: DiabetesCheck

A person enters 8 answers. DiabetesCheck shows its screening score, whether to recommend a blood-sugar test, and the CDC paper quiz score for the same person, side by side. Set `RUN_APP = True` to open it.

In [23]:
RUN_APP = False   # keep False so "Run all" finishes fast; flip to True to open the demo

AGE_GROUPS = {"18-24": 1, "25-29": 2, "30-34": 3, "35-39": 4, "40-44": 5, "45-49": 6, "50-54": 7,
              "55-59": 8, "60-64": 9, "65-69": 10, "70-74": 11, "75-79": 12, "80+": 13}
HEALTH = {"Excellent": 1, "Very good": 2, "Good": 3, "Fair": 4, "Poor": 5}

def diabetes_check(bmi, age_group, sex, high_bp, high_chol, active, general_health, smoker):
    person = pd.DataFrame([{"BMI": bmi, "Age": AGE_GROUPS[age_group], "Sex": 1 if sex == "Male" else 0,
                            "HighBP": int(high_bp), "HighChol": int(high_chol), "PhysActivity": int(active),
                            "GenHlth": HEALTH[general_health], "Smoker": int(smoker)}])
    score = final_model.predict_proba(person[final_cols])[0, 1]
    group = int(np.clip(np.digitize(score, score_bins[1:-1]), 0, 9))
    real_rate = calib.loc[group, "really_positive"]                  # from the validation table in 9b

    # why: how much the score drops if each changeable answer were the healthier one
    healthier = {"BMI": min(bmi, 24), "HighBP": 0, "HighChol": 0, "PhysActivity": 1, "GenHlth": min(HEALTH[general_health], 2), "Smoker": 0}
    labels = {"BMI": "BMI", "HighBP": "high blood pressure", "HighChol": "high cholesterol",
              "PhysActivity": "no exercise", "GenHlth": "general health", "Smoker": "smoking"}
    drops = {}
    for col, value in healthier.items():
        changed = person.copy(); changed[col] = value
        drops[labels[col]] = score - final_model.predict_proba(changed[final_cols])[0, 1]
    reasons = [name for name, d in sorted(drops.items(), key=lambda kv: -kv[1]) if d > 0.02][:2]

    quiz = int(cdc_quiz_points(person).iloc[0])
    advice = "Recommend a blood-sugar test." if score >= cutoff else "No test needed right now. Re-check at the next visit."
    return (f"{advice}\n"
            f"Among people in our data with a similar screening score, about {real_rate * 100:.0f} in 100 had diabetes or prediabetes.\n"
            f"Biggest factors for this person: {', '.join(reasons) if reasons else 'mostly age and sex, which can not be changed'}.\n"
            f"CDC paper quiz (partial): {quiz} points → {'high risk' if quiz >= 5 else 'not high risk'}\n\n"
            "A survey screen can miss people. Only a blood test can confirm prediabetes or diabetes.")

if RUN_APP:
    !pip install -q gradio
    import gradio as gr
    gr.Interface(
        fn=diabetes_check,
        inputs=[gr.Slider(15, 60, value=28, step=1, label="BMI"),
                gr.Dropdown(list(AGE_GROUPS), value="45-49", label="Age group"),
                gr.Radio(["Female", "Male"], value="Female", label="Sex"),
                gr.Checkbox(label="Ever told they have high blood pressure"),
                gr.Checkbox(label="Ever told they have high cholesterol"),
                gr.Checkbox(value=True, label="Any exercise in the past 30 days"),
                gr.Dropdown(list(HEALTH), value="Good", label="General health"),
                gr.Checkbox(label="Smoked 100+ cigarettes in their life")],
        outputs=gr.Textbox(label="Result", lines=5),
        title="DiabetesCheck",
        description="Should you get a blood-sugar test? Prototype only, not medical advice.",
    ).launch(share=True)
else:
    print(diabetes_check(34, "55-59", "Male", True, True, False, "Fair", True))     # quick check without the app

Recommend a blood-sugar test.
Among people in our data with a similar screening score, about 48 in 100 had diabetes or prediabetes.
Biggest factors for this person: general health, BMI.
CDC paper quiz (partial): 7 points → high risk

A survey screen can miss people. Only a blood test can confirm prediabetes or diabetes.


## 13b. Web app: our frozen model inside the DiabetesCheck page

The web page runs **this notebook's final model** in the browser, so answers never leave the device. This section:

1. **Exports** `final_model` (the 8-question gradient boosting model) as plain numbers: each of its decision trees becomes a list of "if this answer ≤ this value, go left, else go right" steps, plus the score each leaf adds.
2. **Checks** that scoring with those exported numbers gives the *same* screening score as scikit-learn, on 2,000 validation people. If the page and the notebook disagreed, the page would be a different model.
3. **Builds the page**: puts the exported model, the cut-off (0.40) and the calibration table from 9b into the UI file, saves `DiabetesCheck_app.html`, and shows it below.

The page itself is stored inside the notebook (compressed), so **Run all** rebuilds it with nothing to upload.

In [24]:
import json, html, os, warnings
import sklearn
from IPython.display import HTML, display

def export_model(model, cols):
    """Turn the trained gradient-boosting model into plain numbers a web page can run."""
    trees = []
    for (tree,) in model._predictors:             # one tree per boosting round (yes/no task = 1 tree per round)
        assert not tree.nodes["is_categorical"].any()   # all our answers are numbers, no category splits
        trees.append([[int(n["feature_idx"]), float(n["num_threshold"]), int(n["left"]), int(n["right"]),
                       float(n["value"]), bool(n["is_leaf"]), bool(n["missing_go_to_left"])] for n in tree.nodes])
    return {"schema_version": 1, "name": "DiabetesCheck", "model_type": type(model).__name__,
            "features": list(cols), "cutoff": float(cutoff),
            "baseline": float(np.ravel(model._baseline_prediction)[0]),   # starting score before any tree
            "trees": trees}

def web_score(m, row):
    """Score one person exactly the way the web page does (same steps as predictScore in the HTML)."""
    raw = m["baseline"]
    for tree in m["trees"]:
        i = 0
        while not tree[i][5]:                     # walk down until we reach a leaf
            feat, thr, left, right, _, _, miss_left = tree[i]
            v = row[feat]
            i = (left if miss_left else right) if np.isnan(v) else (left if v <= thr else right)
        raw += tree[i][4]                         # each tree adds its leaf value
    return 1 / (1 + np.exp(-raw))                 # turn the total into a 0-1 screening score

web_model = export_model(final_model, final_cols)
web_model["score_bins"] = [float(b) for b in score_bins]                       # from 9b
web_model["calibration"] = {str(g): {"n": int(r.people), "observed_rate": float(r.really_positive)}
                            for g, r in calib.iterrows()}                       # "about X in 100" table, validation only
tn, fp, fn, tp = confusion_matrix(y_val, final_pred_val, labels=[0, 1]).ravel()
web_model["validation_metrics"] = {"AUROC": float(roc_auc_score(y_val, final_score_val)),
    "accuracy": float((tp + tn) / len(y_val)), "recall": float(tp / (tp + fn)), "precision": float(tp / (tp + fp)),
    "false_positives": int(fp), "false_negatives": int(fn), "n": int(len(y_val))}
web_model["provenance"] = {"source_notebook": "DiabetesCheck (this notebook, section 13b)",
    "sklearn_version": sklearn.__version__, "training_rows": int(len(X_train)), "dataset_rows": int(len(X))}

# Check: do the exported numbers give the same score as scikit-learn?
check = X_val[final_cols].sample(2000, random_state=0)
sk = final_model.predict_proba(check)[:, 1]
web = np.array([web_score(web_model, row) for row in check.to_numpy(dtype=float)])
gap = np.abs(sk - web).max()
same_flag = ((sk >= cutoff) == (web >= cutoff)).mean()
print(f"{len(web_model['trees'])} trees exported. Largest score difference on 2,000 people: {gap:.1e}")
print(f"Same test / no-test decision for {same_flag:.1%} of them")
assert gap < 1e-9, "The exported model does not match. Do not use the page."

76 trees exported. Largest score difference on 2,000 people: 0.0e+00
Same test / no-test decision for 100.0% of them


In [25]:
# @title The DiabetesCheck web page (first built with ChatGPT, redesigned with Claude), stored compressed. Double-click to see the code.
# __MODEL_JSON__ marks where our model goes.
import gzip, base64
UI_TEMPLATE = gzip.decompress(base64.b64decode(
    "H4sIAIlT+moC/719W3PbSLLmu38FjJ6WSAuEeNONFOVx+zLuc6Yva/XMnAmPowMkQREtEqABULKGVsR5Oj9g92n/xr7s+/kp80v2y8wqoACCFNXt3o5oiwAKWVlZWXmvwvnTcTRK7xa+NU3ns4sn5/THmnnh1cD2Q5tu+N4Yf+Z+6lmjqRcnfjqwl+mkcWrr26E39wf2TeDfLqI4ta1RFKZ+iGa3wTidDsb+TTDyG3zhBGGQBt6skYy8mT9oEYw0SGf+xavAG/qpn7yc+qNr67//r/U6uJqm1seln6RBFCbW0J9EsW951nAWRWMLTdPzQ3m3gMfYT0ZxsKCXDFQEWrKMb/w7A2g69VKAms2su2hp3U79dOrHuo9GsrzyYu7JChLrFoObWl5yHYRXljeMlqlrvbBiP/G9eDS1FnGURkRKxwqj1Jr74wBjtLwxjd6lgc6C8BrtZwN7EfvALPRHoNY09icDe5qmi6R3eDgBwol7FUVXM99bBIk7iuaPfDdJvTQY8YvWKI6SJIqDqyDUQB7u73CUJO3nE28ezO4GL1KMN4nCg7cYWjzzr4LhzD/4Lgqj3i1I+sdus9k/bjb3tjb/3v+UFpr3T/JXLqNlPPIPLv04mBx0e9Ei+afDbU9d97jp0Bvyi7oZB8li5t0NkltvYQtBkvRu5idT30+JUHx18aQXR1G6emJZjcZVHC3Dce+rN8dvjt60+o3GyIvp8ps3L96c4BJE6X3V+qb1pvUKV/Nl6uPp0cvj9hFdg2Z+76tXr1+9fNXpMzyMCwPsffXy7OXLl100mXp44WW722q+xNVk5l31vmq+6b480peNJJqkva/O3rxsvzwVIGocPVtGb/Hora7t2N9Gt15o/TAbW5c0FNv5kx/FV4HnJNRE3k6JnramtWXS2iJaA0zlQ9tJ7pLUnzeWgZN4YdIwYM5pSjfApOm2nWXAjZKFN/Id+/IN3W2886+WMy+2ne/8cBY5WQMBmjRavdHMmy9q7mnszx33hP5YB5bburl13NNjXNVBpaSpmrW41dmxatVGq5bbPOVmAlEDbLltattyW6rtEbftHmmQ7azhsTTsSEO0Qcu22zWAdlTbNjdC2yNp2+G2Xfc4bxv3Ws3FJ3Tge4nfGy2Hwagx9P8Z+HHNbbcxJsdtd50WWt8/ebYaRp8aSfBPiIzeMIrHftzAnfsnJGZXjVt/eB2kPJvUyG9441+WSYoOml/3IcWi2Qygp95NEMW9ZA6Ont4/GUbju9XcA0uEvWZ/6I2uFYPfeHFNc3u9P4pmeEnugcPrfVrpPSwmS+4lzfohhnmkLgmH+v0Tb2W+R7xb7zN6gAl2wFpoRJMJdECvs8AwvN40uvHjFTcZQy7FHsnVRjoNRtehnyS9NjXrTaLRMmncBAlx0wqSk1cVQFhJNAvGltmdemr001eUi71xsEx6XQLp3sbeAmT4JHql12qd0qxouljeMo36C288Jso3LYO5ZPKZSwCG+HVFpGmINFIjp9v1/sxPU/RL/Exg3Gbbn+Md/84fxtEtv9Y7zinaaBFJO9ZWEMf+XCiaxliAUGnz3nKBpTYCNxUmjQURoQiFs1gtoiQg0va8IUiGR4AMmdI4OzsDObiNUHnF92mk/TRayI81JiEJWM/II8zOLf8JXhn7n3rt5v2TJ4fPLNL+UIjPDp/ILxcwVxkjp2k077VKs0iTB7Tz9pbMlZZ4k5n/qe/NgquwEUAUJb0RFLQf94nzg8ldQ6nsHksRsH966/th/8pTY5kHYWPqkzbvHQsnDEHI8YPgCQAvY14IWHoMAmpIrmn59SBpiBTra6fE370wCn3dcwM8d511fxUH4z79gxU1x53Ux4Bmy3mY9GJ/4XtpreMcLz7VGZ9Ojj9DsYKVcDNa9PUg1/j/CMJhbUZFKdVL4Nzpaq0ltBWahd4NzU0Df3ciHaTtUUY8JlbO9AVolpt6Q7NXopUaAcmrAu+0M97h1bDwYvRY4kuZoLV1ZsqtinXTHy1BkLi3iAIawzqKSnKVp3q94XsvDrwGwMVsSC68K9/+sPZecWCNdbAu2cfvp8F47IcfMpJrRvLGV75aV9ULqsQDvOxzQvHcWO5JmXkVLTICuiekMyGkutuFVHebkLqdgju4sQ/8aW3fP/kjmbueVcsF8nEX8ri+UkMrjNfkPb2u77W8iSO4HzDGUxI6LpQgVv8uqwtyAZ3Xms7JJK5b2dURrkpvgMm8pGeTgCL7+HZsWzYNUV2UWuNe0iNtYrUmcV96axDW2rg40kpF7Ak9J2uPO6JzrIyNO8QV4M44WnGXhFePsLp3MwfFeEIo3ntJMPZdRtR4xNcamDVtrcq6vkOKqdlWl4qaa/PeYAWndSgLS2ArUjdXtJ3R1OjK8ueiPtnu7gUp5Meov2ZE4I2ZP/ZXa+IDdhJreyXtyH6r4uC8/053NM3VPFiaqKrAWwmQCq9WlbaPFvlYB2g+8UZpwktrVSXSqnEQupwKXTIoRflJ/zRoUfToH1E7XdavbK9qKC0FJLdRQIdEk5Fl5o54aTSsWdDriZO80jrUtvsauSBkOsOxHV33f6OWUYNoxPwuz0AfwjQlh7fBCgTLcTye+bKoc3cbS5rY+KsROfnKiG2QmVJYLZ2ChUYIZDYHmzQl+cgzrMjIDdyznNDyBs+ZhtWbBDEoHU0a5K2bWDR1kwYtw+K8bjdPTK05hJAkWmsMlK5jLkAHH1eZMsys+EyQnyqVt4OJxZaQTCTh/bE38zCq0TSYjUsWGj+24Mv54djBL/fjDLGW2aq0KgxWLGF+xDNcIEduTlUOXckK+DHUd7gyNNARa6CC/qlabNngWhnhXOiiWz/2xxaBrJIxU4ilR6/n7pFai1ZbOooWuy7qjO3Y6M5fz811RChgMt74ct8KwsUyrTDmIxLB6R1ID+8bXo/BG2VDhuGA+3KtqJb2JvPNsJi7XdNqkIG7Zzy7jzY+NjgVbDHI6PIWcOQTRwHhmeAb2a/+MiE15M8Q1NImEUbpJnMZaM4M4omvLUY10vLIcnP/how4/A2Xc0Q8Rj3YdRS1oOvEnBqxCg+4VxPfssTNX2BR5o/VK2WamKbhmlYybzDxCoALPrOAf5Tj3BbnYh6QCbOVnbd6S1UcPgn82cMe1yM4Sp7AUtvAVCYLHxssDHlCkj1fkBo5Rb3bAJGIyplURjk3Vqsy91FKzgsLryCEbRqkWnWK0tMTws2qrY4HOE8QSObebLbaYLlXRCbI3kEXwzTc7Glli6FjeFF6FA/KSCP0Y/i9WRRoc1BoXV7BKBjpgBW7KJmszXE8K4mgsYeA7k4cU3QNMrY1DMam8AWhscn/omcPLhMGflRWecLsCoZVYCglsqT7JRIflRL70YKXJNRkFt32ZDAKtjVcQlmHD3DxRhVc9LllVRVnstr0NLoWRxnpiSTxxwM7jZfkKFeLxCrR58dFd5wCFdWMqnWjvLRhUmHKRQ/OZ6t6OteNx36ms6F5Rtd3fW1bbY7AbjFZZXJzq09WqA6+dUhupzvrd4PpczF51DQ1/ak4IN18lZV0eTUPivbabQLXojTNtSBNkZ0MQyGLMpAxoN6h6HrdKZoQQhcVuTHw+qrJPh09gycEa2uVxy040Vhzz07r0sK9mkZJuiY2K3RD1ni9wyLjEkOUDEaaE7K1KiUFt89ifa1u7oZ119ywdpWlJeyzwdLqJHkXblRhk2hjmds0clfgcfa5LBC2CpTmJVePIxEWEYZ8PTNaUV4/srTahjyF/TfBUy9OSZrSS0zhDXR/jNxsmTaDkQqQ7JLkdkxqFgzVo6SAjkvku/LHVUaFKE1O+UB4QUI3WdURikXSC7gv4mTmgfGCz9YSrbPwo8XMf0R0utV0WhwyyyJc4Ekk5ECpMwpbe8kCRnqD7YFei6JKp9TNusOTKWaOOFCbqujEhrQGBR/YIVfmVrf7dannfr7C+ReN5T9qDbxYf0wgo3oNqTDZMnaQH5gndVMk6YcOPZBheZOUI8kPjaqFUUjIhH5lCoQG2j39ukowW/JvR/7/XYfhUhZbzZCjL2VkG1IIlMN0f0ksxWSuF8eYd1yuvDCYi8Go7x0XxLqFsU/7WStYmMQo0sBTGP3x2r+bxKjoSCwBsprE0XyVO8oVDPD3GqVW7tMoa2byCZsFxlrmEpFVOR63ZnA0KyKUhqLtiHFpwtRhyLJG5PTgWhC2sE7KkRflcBXBu/Ad1iylrBWKTx4f0+zmMU2oOlSupI9IqJnGkuky5vnHhyyh9fiswmKTwT6OtHndaufqk3+vr34agXKONqfNAJFF+2Z9SS1A6U3LwZUA28MuRCfLtj5uliS2earD3dTZo+JAppOsXjfoeJaTkTLKrM1hyi9nCDahoMg30jF4K8Sa5hSAU7hnBHdLEyagRKEXYrWKwwuQpXEhMmvyULPAYc0MujVtu8PgqqE5eMcsiJHbOEZuYXtSRMUK2Y/d1LF75yeV4cm5D5mnBMODWRDDgz0tpjxammgKXlXaQ/HkFea3nFdYizo3H1ygAgkZhl0siTNVTzCJq12szHTR/bd2jnkrPNw1o7Urac0Hai+OHl17wb3deLM1Wd7OvZuMkXaJ92iAKuRTWZTTfSi3XakyeDV1VDYs68adzqr4IoWgq8wMVaZxitb+USFy0+Z4wdG6Z0m+jDFDMkHt3zBB2uB3jzLWUFn5lzRAGdWGtcePUOlTOckhelwVPH4zti/sKWVmO4ctNwY5qC9ruCrO4JdlZG37uCdla6LDzLGeqz9qS65erfHqZa3Xs0Ahc2O+WK1H3yRXVJBW5YCc0kF4vxBz2yJSWk0lUawjYTXu6nRD8Gb3xCmhABJCXW22II43e+DrkTgCOAmwsIsTrKFTXd8O0HOrRNHIRUHELllto7mV47LJYKEG4cbiN+YvkQLstOwi3jazlp7s6vk9yVSGpTI8WbAVnJZbIrBBvlLGgSqPLNmemUo213NbMu+PSSFnSUeBtskMzdT/miAn/eC2Hi64aBm2hTLCy+ulfcwVF6qvhZssh6utJRJHbcNeoHmO5lRZtdNCwwxkC/3UrFdQ2JGpNgsWq1+T3DEFan9jNI3Au3MqzNictqE2VlaHudWDUgpR3oCR9sU0eQZxgyo3q9R2C6N1FKqpZ0jWgnY2Vf1JQQ13JOH5sB7u/v9QxDyEjWawPN2kjZmui0d7sMe5BwvSeVdwJ1bbkgcNlRfTdTkZJY/zBIgiJM9OOefK2QkzBj2ZTIy+N0qM5QwRFY30mVrxnUKGIQuh6RDDKady+VWUJLK+2hBiaqoAk0SVWlsrfEoRJWIzcPcVPcVU1s6aY//KMZY1EO02v3byueRtDXW6aZFmq+c4jiqLHAglA6M2/VYirkoLSjSVh9XlOhIFnHRhNfCzHGDT6CcvDJepbHAB/w5RQiN7vzb/Ogi9MdbLL6tJMeJzdAfsVYiH6bFJMD55TEw4kzobTMfKJWZ2Zg0rSv7XLYsxheDiJAuXid1fGTDbaG0pGJuq9NCCsjdU9fhgDGUt7pRn/x9d0acAtIta97hJWpdsj3QaLbFZhjMaic8Yoqo1HK92TvBl8QcWT/TulzIZdDjitGQytNlk4J52sBeOi/aCHnCDbITfXHu7uVTWNNHKprzOAyUo4RhNH5M4N0tuKqulO7k40LEGmfwio2Wd755Z31ifnimXkyz2V4XhloQ7j8osg2myTihg+FsT8AgGJo9IFR0ZqSLZyUAAkNcsZWp+czoGKbhtqQvpFfUXSxBma/iWms0Dos5Gn0i1wqzFXjUs0XocbZjN7io8YBWc29G11aXlsmOJAZw8GDGTLO3OMTOgIqFeI9GaZaP5YViZrHi0WbzmNGr4t7TPVuzkorW+qeZpbYuJwrNsbGu195ChXex1zbc1Deu2chlS79r3br271bqaqAwN/eY0deYoVeSgDHzM9FLRcS9OVCnAXK723SHvxDsyQHYK/ePvzvpACYiuk6kCEhQiKZprYv8Bv/w+79saBzerjPZuKy+r13+yFaHyrBsWhAnwwAAq9mEWXzebVZRXK1PbbFgZ2etudzbL7v/RrlWrWa+LfD/o7tbYLJgHu8l7MyjQLhle7Wz/BYOzpp1VVSFQZpWKPqzavyHvL2ercn5CT0pry+6mCnmhACKeqQncOVIhjAl20lJee1f7Tdm6nUfk7HQnVZsfSwbtbhsfM4AbrWdarYspJdCwXtcjcafHHInDHma9q2pLnHfThilslTK3TG3cLHWPbh6q/skL3rqlKBOVUtYarWfVu3bZisYozPqlopeNh2b9UEHIZhJCpfww8c3+LougBS88j42dbYg9Fy29TduOCwgaFUWZM8m7AIjlCL1i8VD2bqF8KBu4qvjhtUpGmdFVoeBhg6gwTMzm+ssF1WNumTWaUv2BYwzPKifG11AleqM5MX5otfmRTuHyM5mCXrtfSAgXssjdjZtTK4rLiHcKLs7GhcBt2SzequjaR7klrIme6cutMrXQlNVbUbfstBGoICebT4M5HcMCvcHAlZjfMsQtYXueVRU93k4k2jPjZLsNihjd874cxywE10/Uq1LHUN6+IaUOOXrwbCZwFBrYAbQc+djjHKlwGV3WV8+cZ1kB0zNdu2R4FMR3OXHy8qPSg3s+m6F8+gILtQyVRUwbjYwt7tjK6iB95ugIhmMGChxTEpqrwOj0QZl8z+c+lAOOOttRLLa7f3J+qM5BOT9UB/jQ6/jjkUxKEpyaghMD9AEyUrJhX1zingV3BIfg5Jv2zg89dQwQlI96GUO2L87BrvoGKTKcvmJZeQe8Dz3rgd6w2D9lgT2wi+f9TKO5D4gsAcz3eR+7elEiqsqvvTgPLs4P8X+g209tdWP3P+VXD6n/4klEPHoMi3aD54OnrcvF4fyIbeEJwIgnbtHOQgyBL+zsRW9oY2dD6jVoL/jAFqpbFfvLL37CDIykfwHyOMg4cQmkGilA+iqHVaAzbdC2L37UxxcVHs4g7e0LOo5p/UwjRa7sD2hC/AauuBCu8+OLJ09MLiHkbCsYy6+G4jqc3uQFId9mLimylPm+rBHhM+M2FyLxXdxf6LsqO2Rn82nJ61QwQ/uCaIt58TCphQIxbV1sPHzq3J9f5O+454e4xmhb5d5pO7J98YI3LFqngARjJYfnWn/z93HoVHbyFM7+2Xbo1BWCgMDbzXHMOso3MqO7oQwKShxGKM3a33G6kCXbJinuCueRWGgKwHIqF7XZyllqm9EG7noL5xD7P5SKSLTsePnqJY34nznDCd7CHKXZy4iiZ5A3AhA3qPURRihkCdC/Lw2Kr/N+XVs/qXjGdlJJtmS8IRTDDGgu3okWhJz/CabdjFZYfAcKW+raklCWMXI1ZIUd7w8i206PXlEW1a4D+wUtxHMxl4oIfwTXN1sKSTSzeGjnh6ptBr84flLKgi2ESoOv1hA61BjtiuSl/2kbkm2FJJrthB5AkQQ17pFsp/1TahLIbo9sdbgbdQ7bZ7bE76aa1Is3PmJCuThieL8ebEuD/a4C6A7EM1nbpNs3331bHqzsw7apbBLMNg/sCnp2FArfQH9bc9y32GeyaoBX13htILAyx0pgGeeMGsQd1LWiBGGpKIQow9CPbdnMNo/GfLbeCEYTMIaEAaXa+OF9GthnpzaEi7/AnLgt24IlMvKn0QwKAOvEvXKtNgo8CAuK3V1cXx3O//v/YFh89eiFx3v10ugKB9Yp3el/AoQxhbwn3izRd8mdhg2XyBv2xV8SHHLIno1FaRGJhG1ap0U6MoCsc9g0bImYrdfbVxCe9srhzLoIBx8qsVUwIf6CxxtNCOYjAgAwiXqvGOm/QF7x0JoNH7AYDEA4NBFRpTIwoeHFaA5o11cGgdRKKNFpw8jzqVomD/LfJN3Ccyr4VWKroyLt3vJsOtaETwFUjDZJy0y2HY0g3IX1C2icVaMRhNBdSYZIED4Okdnw0Yi0jksU+ZtCZUHOQo4K8UcRlUdPqGYbtQoeGMto/vixnGyYXYqxBOgeKjYb0Gj+ONpeXz0an5MNpL0OZhHKJeaJIdp+JXW3yz/IPot3Ihqz4C0WszuRamzPQXDnazU3guPYeIevRP5At9FRrXoOYZ+tS8CysjMu1cUXt8fe+t4M9i4GlEbxXTURt5snbzE33/y4zULpKrCvEWSCxYrt2mSBw8+HfsDLyiVgebiM/ee/gxWjcFyzOP7uJ7/FiimBzeyj76NKM+bxph918BKrYht1jx6g7gjvw+DHOV+z34u0jOLvQdwC4C9P3j/54Vtw/zbqHquu2K+bymKB74zWfuzNdvcH5NWCS7CO7++1yIk17sbwRWfBxP81a/zH6V3ygjZ6o8x9G7VOtMMU3sE/Q/ViAGkZhOyjLpDOszpNQL1Lnlt/82Z8rPMI2hLnIf8OjFnA+UszZyXwL8+gl/PoGhpzC8VPtfdHLcdUFnhgjQIENKiKiVwXkgYxT/zvsfoVgl+avCWwDxHWHE+miSkEbl+8JP6ikIl/44cU9QE59m8o2hqkrmjhR6zCyFyCBQMCNV9zcjZyA0LWPb1y6fvW/E6X1MscVgRY//Vf/zMbnbIr8s7Miec8XFWMVkJ6/PRRYdnCnwqjzkj8GX2oGxdNK5pYmg+fVNku69YRhZy2W0dPlL+PhmouzlWIXwMTego8/VsAKOMRxY8I7yJ7k1KU51C9bsTFOHxOqYLMPJRUq2l7suBXU+cltMAsSaPp6JlpXWY5OGUD5tdVkbQ8q7hJcOe9YQVf+2bcrKo9d0qGNY77xxZScgS/oytrFnmU6KkS/SZCqivZ/i3Q1FjVNAXzqyJ1XlicRwMDmITZMPuFzKa2kuUC4+FEpzFil1IqfE8FdQtJEg4p4M4dIsWIhvplMuWR23UMKGFqThBfX7wQXURjQfp9pj9doGK58o0Bj6KN5BQhvFfdg8qjqqC7vijzdZHsIpW1jMuzJMiDwo4bS67krUfpShVYxzEZMJf1pckSxdfl1VeYIWC/Pvt5ePiQl4HOJuAPZQjyy80phTw0jcxDaYEOOR/F8kJlfzflsNZzCD+RyQrGAmiHQuhqeokHmMGt0Cd7N/Fudah72hY7x2o5OJ0GKROsWEwbeZgIVVnpbWShlotMDbQsdArZLXmbBMrHQpXlSec4ex/1oYTIEIF8CALpm3gPN1TkfT+BbQO+4Ai8a732UB9KUwcfkWpDJD7tWpeqclTKTOjlucTFPLqNM37ltYQ++hHm7GUSrJBEN+RPFVNJoeqW6Bds6yihDAv8+2k03hoHkyZKtlQHw0pJvIdDYhrmaLwpIkZZjZyySEejmACfMqlXxxDNwRMdhfPk1wbpJZzCRE9QE8D2KBe1OpZUrToW16XyPI3wvQ46CrNCyJk/N2DEtatqMfDPalteQhOFJWxxYplWU1WdrUqkmtqASj9BPB6ICpOQ8Ah0GBZptQQijILgmH3JxG0O6ChBTScJEEDopHZzgwuxdRy5JKtA9jumdgFZR3zZMJLMXOgjvzfeCVGZOiB6pgCedTsk1WVhb48X/Za5kDrljWNUEo2oDmE8JXUsmFwWJ0O02jgYh/sy7J0GzXxqX3Q6pwpq68RpdU+/0Lg3MpmsCdXlK8GZQw80gWruduMugWRfHLU7lWbK4caFJUW6uQnROT1FWJosBuEEFxqhg1Mo5N4yFFZi1BLDyOAv91hAVGsAWeiOTraOOYmtXNlc3HPhQCS5Y5gmeJ2sh558r4imdBrEbCCRvrojhYDcclJegfJdJDJkkPoGc1m0rYaAe+l+wn1iesd83gdEFAQ7yarTk68JMG0RpIFUqXWlzcX7zAzgLZp6ypO/o5JW2ehb2MXDZTBLDVV8SVMu6tKx1JGtFtVKJunsLtfBRke6KMvUa9VW8SuojyyLddE+wgceTiGThgjDXvzlUmluHXYQBd1uto50+EYZdrVv3r25vKw71k3gccu/vPwWc8eFkwiFuqUQ7RZ82MbOEZJGzNYof2iwVWlfnOhIkkV3E4Xun9TmPxzQBB6iScWk8KoljARTQr9lljJ8m1rxMqQRsu/C0eiFtkh3Q/nlks7jnuRIN92uJuGPAU8WEFFlAMQopLJ1UQazIzEthS1o8qMFwmFjktjCnJh3PNV8uStO77yQQ0Ifl+g2vTNxO20r3F785d0PL6WkwsABO52atP6wKidgbMdqukd0jdIo7OyxJtjgWpeVWjIpEhy0iqE23ZPjIqKVhRNS2Vfgz2nn4m+0cqkmwyPhN0aEAjfPl1mOGB8BI+PoKmRrK7xj0e9aP4Szu0I5DAGgOFhgvvgOGwcNs5++gPZDafDoWuSc1eoU5EER0r/jyxESC5IdDDrkL3RRa0JpHXxyjQxfEVlMM5GBAX2nakoGBM6PJvIqi8yiXYbEJYpvmba65Af/JobjtobYJSo6r1mwaQkpKOkiGo/tldmkQWuTbHEkHXEwmVVY6bS+Dcjnh8vZOtfxZBH2d2aRzlXljH0f3VJTQUWsfhbwzC9BmC86JpRS6NyEVnNEnskNVE95sATNFtXwHwQG7q6t/E4R/jw05ezgOSFaXoW3TFBm3YS+HDjWGCkHhjchYlXSKNfJTcKzR1WDxc/S0Rf14PS7wShxlyPsxxovD6k3cNjh6VnrEFb6gebDAxGkB6iLQHUaSAILG4W+V/Shwp+HwOhafSgujFgyxGLKZ2VhKuf0bfa6VYPsrVPl38Y53KzK1AMp1t+gupThIZr+J566OPoFMERKZF8SZAsZtyq+J5gZCtZPCF2TqPMs+pQjppq8cXyocY4No1iTMru0PKWATIfFBIm/0mnQmJwX31qU4LAukZ6JZtZb6HwPK4U4uX1MRWJocKAopWFkUWQZKqlx/vCilEcht4ogAxXpso8mHhclLYnIoNXhLwkFWn7++bsfXr3+88//dvnD9z//DNAM4kLDunhSmyxDJm4N5cgYIaoBwtTVP17PfL5mEv8Z43VRQ1yzf0lsfH+NNkQ0sv9k2fSEKYldJd4PjeWJSSOfSQSJaR2ZL9JXKxQSmvt9CQIlNWVSqNVbt6hkGmHd2lO9nj9/5rFDFOk7TwcDy46GNN92HT3TxrbQv7Vex3EU1+zXtI+ADT9drMKvuTZ/fw5VLZjUmFzyxBpYK8r79qz3rbZzdvrBsVAaRldOq4MLVDX13mPvFX5KatC8olyWvjbTB/qeSkb1AOyIYHEIXB5a94QJiciaoIPkAMlapoU7QUU88qaKFhpjCZ2rsbzHC4D5HlVEDgqIPgxkPHybCPb+Q59fJUIq4vH7oFyWtUezp9/zbzdI3tBnRf0aN6rTI/51DvjZxQU6oosaOiE4XOi0t2cA+RaEv8JX/ARKvWJqfpzRBlTLlxmCllqIClBzxCSxD9DBga2miwrckftYxmGJOu7cWxAqgwuTJvzRwIzXOKY2Si9JtFYympCW0xaDasbUzQkX7CQH49wOBJFsT2lxKskazOeSbUM9kfQ+154NmmSlwi5NBk2ZKISLoCBqT6n9e27z4f3RB/2iTOTBgbxzQY2wJyS8SqcVNP425JEoJcdthZQ5L4VUq2F05SjuYkK8p6fvmx8+6HcE5WyWv/e+VzP8vMZNjz8857/tDz3+2/lQ70mL8wHfaK03ENj3/C9IemBi8777Qc98MKnh6cWgWVc80DqstQ6+g2TFBuNFDVuSbuvGqvYH2SN60lcv+fSSX2INsXT+ByybmmcyA0TXwHMhBs4H3efNnvp5/Lylf54+b/c6eZ9SBodXsB4uBt3m805P/e400VL9bh8BAM+1wgndHHguBAz+FdlyUPNcU44MBoMmvVQ/kC4K6PtEXzDrFq5mg2JQsQTKLM2xxAFtZ4oVD/soaM95mAH9PIR3QkfLjPxay8G2oPoKk8OPLgbUvs5gDg76NG8CkidDtq/KzyCsnTn8zJy1UTSFEFCrCho6GMrHEt5fpqRiavIC8wS6fCrN62t8nxl28K8M85ZNLJgAS1TXewEChTO/qAloSwxFvAcr0gQZolpr4KbT7tYdpQCaTib7m05B7LccLfHXYKgHTrvuKDXQvM8xoBRpIr2zUNVd2cU6Gm3g2zkGtlELkj8uYAWbDUOkFEeWwLczRG1V9qA8aVtjZ+NTqey6aZgGtuoMksEPrH9dAI4DP6llZKyzYK6RNHZYCHyoDy5qK7rkKG2PhysKTCLjPeaiRgWnrlzX1fkafqPHEO/r9/U6DiqZYeC18eBi7AogOJbNdt2lCH+t5jlDdDxUjxqe+lFXPNzEZBjLccU4OLD1bvyfYEknVLrWU+ytOBMbmCYTJ82eFm4zkzrCm++wMHvy042G+DIPPIefwdO+en5JG+zU89Ah56pXEEdqfTqK1Pe08hHMxrZGtxCVH6xMmjlaJjgGsPs1I67CNpOJ/cMgASm1fQj/Lr675A8KYXEBGesP3OA9pqS60YvZDO2wTAXcv7/+++XgvTA01+U7XCTu6CIrJy8IcorlF05Wx+PowoEM6os/vSaordN//ef/atPHnttH9OsMvzpN/OrQvQ7d69C9Lt3r0r0u3evSvSO6d0T3jujeEd07pnvHdO+Y7h3TvRO6d0L3TujeCd07bR7kuLx9/eLPP70FNq8/jbANBQRBi7+Sh42vgY9pGPLnjRfENMYoMgYie+4GalrnFGX5jhKAe3v575q9aceeer1u1+VVP+k/IVHOLDkIlzOoBNpuzz/BAn+o2V9lOxjqOHgYy/7tT9/9eUD0lAU7cwKsl/21sg6cF7GtsoNmVpd17B/UgoNW/WBfpz73D2YH+6UKj/26+wuCODWbhDDhZVZSmagJebcjtw0zzUS/Hjs1VeK9DwhXlSvCM5XTNigZx96dS+5+bSW2GZ1Wee/Ufla4ZwH4RZZ2wGHtPULr2THQwuEfONNArk/4WsL0BZQEox8HtAwVCnyiAkQvWOtJGt/lBgBpvgG5hS4EQuLXsmULx165fd/cfQtnr+Rq1l06Y+ClbKavK7VLwGAJIMnp/ayiQ3ACWp8/84PMKBdMHbuOhyQB8hufPz/lpmwRFy6UKYt7ZXeE24iAra8rfColQdKCMRTGp/Z0RVNVKJsoDGlg/1DYwmXrV4zwbvEFw5ZXyEIkc2S0hm2zD/UmJRsTGB+UpYi4eMOGmZRbc8gwjLXbUTdM0cFKO4raTySrjOhaXyn/rnY9UL5YfaUcReZU2pFSd5n1gXgwr2EfN9kygxtqbz8nwdATetdu6PwFFIaHS79/b7ico8EGdbCvHAVeZ/sH11hTH/R33PbR0fvrD4PRcwV9JFjUeyKLCi6dVzBqg+SF+j5h7dq5qa9UoxvwEr27t7fmrtb39mrX2hn9/Ll2czFotff2bs4H+FxNvUTjCSnuIn0LhNf2cChOWSXV5UZ0PSjgSgOu9zEZ0XU9hAksrT5uJN/7vCJQUQ9U+7i399EIw8gGnZqtP9loOwDO0/MH5lX5Pg3mGHhSiUKtlrKgSStgIE7kBOehGN3Z26req8it4YHN5V+2Wvch2OUUSol4WDEdCRmlxAYlP8Sr9ynxXovpF3WU1YXVXVXZRtUF/eLiMRqZqPg4nj3BYZD+BkhcVcAUuccxfD5hSzoPk0S5x3dc4lUTkilOCondcpags31fUsVQjRc9D+/HnJ4LpucK58WStHYROcUWX6Qa0rsaJDcmpCZaGF4iS2x7TkGzRQX9KVWOCZBe+vdFT7SALaGQK+4cwWY2dUYNWt3oCjv9YbpCKMpRGHbeXlVkmarV/uLVWXY/x43Lr0oy8NHFWJlQ1jVXRQaiJ6pGsPgAlFVae5QOPsHz+PQMgwQ20Zvgky+EzEif8erKcEaBpniuXA9Qi93cqGdQhuPKHsV48JRm7PNn+tfl9yGR+G9fz+aKL51YxHmJ7eiemgTw2IsU4hr1MLT0s9IW2+GmsjaNIkIvlk0BMwoQFGa9mks0Q2oucWK36PIYo0NaQkbBgQj7jS9cgURfS2c6erZs/zXzf/+gAOM/7Av7QKMsjZWhZe/El/YBej+wM54ssRvVnmwqnhPOkSG8ezcgt0vJ9q0c+u7d81qZGM/tb3wcTs5JG07PUbgPe9PSfyyRnzpLVGrAPnj3zh1Sy2ywrpAGdRO7vMu9Gu/We/YbShvZB5r5yGPE5P05om+xqbAIBI6VIIOJ86xKCSY2mwwyiEWQLaT+TVGCKtKUl9NT3FSHahR4iEwFFBMmNZu+NSIsd2NM4hoV94t1Q8R7YtlSBQ0FJ6I5lCTsz/0epkD53DgVRANowK0/LkERAP+Gg58s/oZmnkIlOu/3qlp/Xyg9gsi53VchAIKgJHDs6EVpiGjFRLmUuJmLTagUo5un9H6WzX2JjlbdzGHX1m7mENCYu9k5AmdaGSlzSImrKBmgFc/Iz6F/xecXJwhcNrJ34YOniwEaNtYbOpZa0oN0cZA9lkOjbvzEjHQwp/YMCQeee1bVNe6FDQW2js55WssvpotD3QQtbq7Kz4HCIQGq3xcoOkSVFUs2J3RGsySz9fYL9dRy5D08IjR5vm/tH+Bvb38/9+HMlqp6HV4d/cg9u3I7Pv654n0Sy5lnJof47B9kIcyWE6Lbr+2L0qEdZSghYRAe7BelnmT69vsmT1VxXc5hOPp7bSWh8GY8+js/oKDRxeAIJKezsgfq/mBwJ3OtAoswLAZPcWtvb+O6MjhQ3NwNIt/uZRpNyVZTtKrUR0CBzHd7e0ocXjSfG9zA7x+qR7k3UBA9uRRhS7PADcYHkIjGqsF6EZNR4E8lIFnLaTurjsq/NATeAn3AW/h3X7EWct9DyXnYjRZ1JU3+5KcVR4y4+z0i8/P9lzPK4qvzO0gI4cl+SeTwmVQkeFzqiIum1geivkmkO1ZJGMg/pQv3D2SmTB5TmexHqEkA5+na2+M/yFA833eUUsp74mcH+/9YNpvjk7wfGiLNvjVc14+KilLrppUdb1z0wwgVp1IZ56kDXYxymeysFmzksb6LMo4kQ0TV1JGEIAQnrEvFJs07CSkeMHP3FcXUvIAdpAFMpV9yfVHA2bW+LHG5hgRFNxRCN2oCMUI6CDAKOQmR4bnPpUNfFoMi+TFJXGU2jvwklJtUckYWQ0oEZXwRFWI0qfpkku85Te7mC5ycliiWzRZU7d3zsrAu1aNnORhiZRL4+z8WB7AvUs/Zp9/lhViXV15goVLEmkuD9h0YSzdXDnEYP62wz6gNsyW32leSl5lSrbTlLN//lWaihJ7MSkWxSqWwRGH5WSX1ofsJiJKwmaPRps4Lm668K0uPUgYZRqLK9F2xGLX8QAMZXnajnim1DOeyjKRz6atP+yqgolRglQ5b5lXRfMQckDN0BntABT1ovox5lKpT+lGsTC5gLPXDf/Nz408vEYgGImXWX4GcxOMkSRACsrjoh4t9JHNT9CDZNZMin2KRNGb44fkuFhVunXNqAMT4wPGkarpFM1fMuH5wiT0sPHY85t+qytqYbUUvvMTaHoxCfxKpX14mpGheqa9mZDe5rI/XrPVjsajPySr5SoWDW4i1z8Vb2YXBcfQ5LtpqImWTUH9jVM/hHtV3lhTYi+R6U40VC8nQetF6aWgEFllUDk6zjGJvdSpXUpbwcOGobjbWEndrHZfqheBjMwidsx7Q9o2CyFN0YUpmAjUX19+r4sAlKuQQ27PkAD7vis9gm0gx4q064aKUwqVhqjSXih5ouWqskQJ91XGIjzvfAmqAivm5Ax1j2e0cvBIgPqaRjrbDn18FAAOn82Qu6dxQi772Vdrrs27+TPggtl9b02fYe8rkzi3qaMl+MMkm5fdxfOuFPsbyDX0st1ZDzG/zkxUFuyQeSNJxkKX6IRcdo95BaQMWl3DUv0a8P4++8vTU8WnxEVLR1wOCCyq9gjddYycsayjkL7ZUGUN+lMc1hM55S74ueFy6B3wlYCUvVTn3OkDErr1DXxTYFB2IQnqksJHDPn+Kais+WtbJTv206dhP+15nkahzmY3pIIeKb3lgRqZ7e1MEX0fLpEZ5bWzvRl6bwPZIg91TILfgPmFANT0Udf5m3VW3+8UQcIm6WSqg/xD6KsAro+CyCKx9WwK4T8x+UTT5mjAmIlJNRc2W5Y0Im3Smkp2SlllvzQmVcuPNwNXudMcHR4ICvqsI9sqfeDLkbUkOeUBF5pKjw5JDFmNw8XQ9qaEnjhrXzXLEHZMb9J7Kb/DLW8L9jI9K5eztcbJKJX6e0z32ofiAxKGfbbpstVkanJ1ywE0C1zpojfDZxxJutvsxLPaKGNokiMH4diV+5TiZZX1UzIJax+iv2DBbWz3ALA6fKItSHS56VEtBU3EyWENROAGHsu/tTbYuB6md47CJUS73VGVqthA6r5Ti/QH/+s//nXJOUrwWvTceuxf4CahNYfJQhfg3EUchIskWvbwEJU7K6CVIyNPyUSyuD4GsXEEkyZBlqQuTq9K0T1Ip1ek6VKp76lCNbkuXTLXy2qhWsQ6qmRU9dXWVU+teCzcjWTrwPxHnbUwAFjKuWdZU1ZE+Jl/6PitL8D9hpalFAvBBHZ9ikVyqSpY9gqZU7EOLZerPyH5FcQ9lNLNj75RcMQ4AfIjwOhOs3wNnjjIdoH/116FWZDL0KYMoM5MIdvZ+XZJilNh0CVf1OR4juTmkUq7tmPIgh67a5MBw+lsgIjf0qQpLtWk5Q/ITZnuIYuZ8jJzhVyQg2JR/JtpmDdT5bmuN1H0xBbLWcg7ZDguAZnI0d66vFA/wgKVvaNX5oKYS7rxMU83Qnz83689a7QPjYRCaD+vPcKp+t399NTCa4AA91aR+iK9/NrvH7fZxu3V6xNxI6Vbq0XgBp9TpF0qgcF5c9uQ+X8dxnPGUyBMdMa+N5sjiN5t7eziZbtDu4sf1FVXQ0t/zQecIKK/wRlGmiRFOrvLaEZWQXNR8k7gSfCACNNI1nK9ZAxaHZLw9e9au59nDVqYQ0f681f78GX8vUGuwjhDFX3gpBrzPi3f28Ud88iMPVPk7tstAj0GxAc4DuBrPsjSoKbzwu5+v1LWmj1iias++aSxlAsZfO5ahBwE5CciVYs+qQfvYRI9gcNjbyOc+WHygvDF+2mqIgql/QgDkZYg/vb78abAK8Zkr7Nl1sgxG76R5fObIDpSV7I/rYR+4w/uke60T7I2+d+BTZw9bnU5LPz3tHjXv7/tZPVdMmWEjNP3pkHp1QzLXm1mV1dsXrwZoW+NnGSJIc3z/w0/EoM0GWhjZZDn6qjbXJq4gkoF4P/+gvp1VF7wKT/iOrlhfyd78HuCr3e+O+sPNHLW5ugdEZKP4fV+nvenwA15YfFACxkJ/HyxTozo1SBlbnRlkr5ecvRpI5SeAGQVnRjk9uSNzM5E+yOhBS+ZVXlcxlroKs9AIh8yiPEJ9gey5rY4m6AXnoMBzWx0AIJcHI6HVc1u2yPdsW+91qY3Fk/keepZS8JRXGu9QvSGcAMYHUerPOhThkloOAx5Bk1oTM+ExUNOX1dXoUxXMVIYeV95IDajYSG4i6S1RGPsgVbeqEr9rxwDkRWzq6IAicL5XgM13Hgta7+ovDU/u9lVtUvGbemUlvoPKzZW4OlNkMJiTBobT9TD8rWpU2LQMn2HLI3UiSrlIhC39m8zbNA7fz4QsVampA9f7WZvsNJ1is+x2RjL1SQR8V8UbGiNKsU7A12mGMJ8fPxjc1NMqQqpvIEB680cQ6n2ui9LFQZtai/H1GF/+Pt9HuJHcfmY8DlHJJdtJwSlIkiUg8/vsNPwPNlu9Q6jQdS82o/2wQAFgTGhkTxXdcaugrehfY0PkofqCxyF9n+Tiyf8DUjmo8bqrAAA="
)).decode("utf-8")

In [26]:
# Two example people: type these into the web page and the scores should match these numbers.
examples = pd.DataFrame([
    {"BMI": 34, "Age": 9, "Sex": 1, "HighBP": 1, "HighChol": 1, "PhysActivity": 0, "GenHlth": 4, "Smoker": 1},  # 60-64, male, Fair
    {"BMI": 23, "Age": 2, "Sex": 0, "HighBP": 0, "HighChol": 0, "PhysActivity": 1, "GenHlth": 2, "Smoker": 0},  # 25-29, female, Very good
])[final_cols]
examples["notebook score"] = final_model.predict_proba(examples[final_cols])[:, 1].round(3)
examples["web score"] = [round(web_score(web_model, r), 3) for r in examples[final_cols].to_numpy(dtype=float)]
display(examples)

# UI_TEMPLATE (the web page) comes from the hidden cell just above.
page = UI_TEMPLATE.replace("__MODEL_JSON__", json.dumps(web_model, separators=(",", ":")))   # put OUR model in
with open("DiabetesCheck_app.html", "w", encoding="utf-8") as f:
    f.write(page)
print("Saved DiabetesCheck_app.html (Files panel on the left -> right-click -> Download). Live page below:")
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
display(HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:780px;border:1px solid #ddd"></iframe>'))

,BMI,Age,Sex,HighBP,HighChol,PhysActivity,GenHlth,Smoker,notebook score,web score
0,34,9,1,1,1,0,4,1,0.865,0.865
1,23,2,0,0,0,1,2,0,0.035,0.035


Saved DiabetesCheck_app.html (Files panel on the left -> right-click -> Download). Live page below:


## 14. All results so far

In [27]:
pd.DataFrame(results).round(3)

,model,split,n,precision,recall,F1,false alarms,missed,AUROC,PR-AUC
0,Baseline: always 'no',validation,50736,0.000,0.000,0.000,0,7069,NaN,NaN
1,Baseline: rule (BMI 30+ or high BP),validation,50736,0.210,0.868,0.338,23088,930,NaN,NaN
2,"CDC risk quiz (partial), score 5+",validation,50736,0.237,0.807,0.367,18320,1365,0.758,0.298
3,"CDC risk quiz (partial), score 4+",validation,50736,0.195,0.929,0.322,27088,504,0.758,0.298
4,Model 1: logistic regression (21 questions),validation,50736,0.311,0.762,0.442,11935,1685,0.821,0.400
5,Model 1b: logistic regression (8 questions),validation,50736,0.306,0.759,0.436,12189,1704,0.816,0.392
6,Model 2: gradient boosting (21 questions),validation,50736,0.303,0.790,0.438,12821,1484,0.828,0.432
7,Model 2b: gradient boosting (8 questions),validation,50736,0.300,0.783,0.433,12948,1533,0.821,0.421
8,FINAL: Model 2b: gradient boosting (8 question...,validation,50736,0.264,0.868,0.405,17101,935,0.821,0.421
9,Rule (BMI 30+ or high BP),test,50736,0.208,0.867,0.336,23311,940,NaN,NaN


## 15. Results summary, limitations and sources

**Results (test set, 50,736 people, run once after the freeze):** our model caught **86.7%** of cases (precision 0.263), missed **943**, false alarms **17,148**, AUROC 0.820. The partial CDC quiz (5+) caught 81.2% (precision 0.237), missed 1,331, false alarms 18,450, AUROC 0.757. The simple BMI-or-blood-pressure rule caught 86.7% but had 23,311 false alarms. **Compared with the CDC quiz, we missed 388 fewer cases and sent 1,302 fewer people for unneeded tests.** Validation and test results were almost identical (17,101 vs 17,148 false alarms), so the model didn't overfit.

**Limitations**
- Several answers are "have you ever been told…" questions (blood pressure, cholesterol). People already diagnosed with diabetes see doctors more often, and the data includes people who already know they have diabetes. So our scores likely look better than they would for people who **don't** know yet, which is who a screen is for.
- Our CDC quiz version is missing 2 questions (family history, gestational diabetes).
- The data is from a 2015 US phone survey, self-reported, and may not represent other countries or today.
- It misses young adults most: only about half of cases in 18–44 year olds were caught (52% on validation).

**Sources**
- CDC Diabetes Health Indicators [Dataset]. (2017). UCI Machine Learning Repository. https://doi.org/10.24432/C53919
- CDC Prediabetes Risk Test (CDC and American Diabetes Association), https://www.cdc.gov/diabetes/prevention/pdf/Prediabetes-Risk-Test-Final.pdf
- scikit-learn, pandas, NumPy, Gradio, ucimlrepo
- AI tools: Claude (Anthropic) and ChatGPT (OpenAI) helped write and debug code and explain concepts. Claude helped plan the project, write the evaluation toolkit and comparison cells, export the model into the web page (section 13b) and redesign the page. ChatGPT built the first version of the web page. We checked every result ourselves and can explain every part.

*This project is a prototype and is not medical advice.*